## Encoder Layer


<img src="encoder_v2.png" width="240">



### このlayerがやること

各トークンのベクトルを、**文中の他のトークンの情報も取り込んだベクトルに更新する**。入力と同じ形 `(batch, seq, d_model)` で返す。

更新は 2 ステップ。

**ステップ1: Multi-Head Attention(02 で実装済み)**
各トークンが、同じ文の他のトークンを見て、関係の深い相手のベクトルを重みづけして取り込む。


**ステップ2: Feed Forward(03 で実装済み)**
ステップ1で文脈を取り込んだ各トークンを、1個ずつ独立に変換する(他のトークンは見ない)。


この「混ぜる → 各自で整える」が 1 layer。これをN 回(論文では6回)積む

### 各ステップを Add & Norm で包む


$$\text{LayerNorm}\big(x + \text{Sublayer}(x)\big)$$

$\text{Sublayer}(x)$ はそのステップ(Attention か Feed Forward)、`x` はそのステップの入力。

**Add(残差接続): `x + Sublayer(x)`**
ステップの出力に、入力 `x` をそのまま足し戻す
→ layer を深く積むと逆伝播で勾配が消えやすいが、`x` を直接足すと勾配消失を防ぐことができ深いネットワークでも学習が安定する。

**Norm(Layer Normalization): `LayerNorm(...)`**
足した結果を、**トークン1個ごと**に $d_{model}$ 方向で正規化(平均0・分散1)する。
→ 各 layer に入るベクトルのスケールが揃い、学習が安定・高速になる。

In [1]:
import sys
sys.path.append("..")

import torch
import torch.nn as nn
from src.attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, src_mask=None):
        # Multi-Head Self-Attention + Add & Norm
        attn_output = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout1(attn_output))

        # Feed-Forward Network + Add & Norm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))
        return x


In [2]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, src_mask=None):
        for layer in self.layers:      # 層を1枚ずつ取り出す
            x = layer(x, src_mask)     # その層に x を通し、結果を新しい x にする
        return self.norm(x)  #全ての層を通った後、正規化して出力

In [3]:
import torch

encoder = Encoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)
x = torch.randn(2, 20, 512) # (batch=2, seq=20, d_model=512)
output = encoder(x)

print("Encoder入力形状:", x.shape)
print("Encoder出力形状:", output.shape)
total_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoderパラメータ数: {total_params:,}")

Encoder入力形状: torch.Size([2, 20, 512])
Encoder出力形状: torch.Size([2, 20, 512])
Encoderパラメータ数: 18,915,328
